In [1]:
import pandas as pd # main data tool (like Excel in Python)
import numpy as np # math & number operations
import matplotlib.pyplot as plt # basic charts
import seaborn as sns # nicer looking charts
import warnings
warnings.filterwarnings('ignore') # hide unnecessary warning messages
pd.set_option('display.max_columns', None) # show ALL columns
print('All libraries imported successfully!')

All libraries imported successfully!


In [2]:
df = pd.read_csv('D:\Technocolabs\Delivery Five Cities Datasets\delivery_sh.csv') # reads the file
print('File loaded!')
print('Number of rows :', df.shape[0]) # how many records
print('Number of columns :', df.shape[1]) # how many field

File loaded!
Number of rows : 1483864
Number of columns : 17


In [3]:
df.isnull().sum()

order_id             0
region_id            0
city                 0
courier_id           0
lng                  0
lat                  0
aoi_id               0
aoi_type             0
accept_time          0
accept_gps_time      0
accept_gps_lng       0
accept_gps_lat       0
delivery_time        0
delivery_gps_time    0
delivery_gps_lng     0
delivery_gps_lat     0
ds                   0
dtype: int64

In [4]:
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct.sort_values(ascending=False))

order_id             0.0
accept_gps_time      0.0
delivery_gps_lat     0.0
delivery_gps_lng     0.0
delivery_gps_time    0.0
delivery_time        0.0
accept_gps_lat       0.0
accept_gps_lng       0.0
accept_time          0.0
region_id            0.0
aoi_type             0.0
aoi_id               0.0
lat                  0.0
lng                  0.0
courier_id           0.0
city                 0.0
ds                   0.0
dtype: float64


In [5]:
df.isnull().sum().sum() == 0, 'Unexpected missing values found!'
print('Confirmed: No missing values in delivery_cq dataset.')

Confirmed: No missing values in delivery_cq dataset.


In [6]:
df['gps_synced_accept'] = df['accept_time'] == df['accept_gps_time']
df['gps_synced_delivery'] = df['delivery_time'] == df['delivery_gps_time']

print('GPS synced at accept :', df['gps_synced_accept'].sum())
print('GPS synced at delivery :', df['gps_synced_delivery'].sum())

GPS synced at accept : 1483864
GPS synced at delivery : 1483864


In [7]:
print(type(df['accept_time'][0])) # will show <class 'str'> = text
print(df['accept_time'][0]) # shows: 10-22 10:26:00

<class 'str'>
06-04 11:05:00


In [8]:
time_cols = ['accept_time', 'accept_gps_time',
'delivery_time', 'delivery_gps_time']
for col in time_cols:
    df[col] = pd.to_datetime('2023-' + df[col],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce') # bad values → NaT

In [9]:
print('After conversion:')
print(df[time_cols].dtypes)

After conversion:
accept_time          datetime64[ns]
accept_gps_time      datetime64[ns]
delivery_time        datetime64[ns]
delivery_gps_time    datetime64[ns]
dtype: object


In [11]:
# Example: How many minutes from accept to delivery?
df['accept_to_delivery_min'] = (
(df['delivery_time'] - df['accept_time']).dt.total_seconds() / 60)

print(df['accept_to_delivery_min'].describe().round(1))

count    1483864.0
mean         111.1
std          219.1
min            0.0
25%           43.0
50%           72.0
75%          119.0
max        47739.0
Name: accept_to_delivery_min, dtype: float64


In [12]:
duplicates = df.duplicated(subset=['order_id'])
print('Number of duplicate rows:', duplicates.sum())

Number of duplicate rows: 0


In [13]:
before = len(df)
df = df.drop_duplicates(subset=['order_id'], keep='first')
df = df.reset_index(drop=True) # re-number the rows from 0
after = len(df)
print(f'Rows before: {before:,}')
print(f'Rows after : {after:,}')
print(f'Duplicates removed: {before - after:,}')

Rows before: 1,483,864
Rows after : 1,483,864
Duplicates removed: 0


In [14]:
col = 'accept_to_delivery_min'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = (df[col] < lower) | (df[col] > upper)
print(f'Outlier rows found: {outliers.sum():,}')
print(f'Valid range: {lower:.0f} to {upper:.0f} minutes')

Outlier rows found: 126,528
Valid range: -71 to 233 minutes


In [15]:
df = df[df['accept_to_delivery_min'] >= 0].reset_index(drop=True)
print('Rows after removing negatives:', len(df))

Rows after removing negatives: 1483864


In [16]:
print('Number of rows :', df.shape[0]) # how many records
print('Number of columns :', df.shape[1]) # how many field

Number of rows : 1483864
Number of columns : 20


In [17]:
df['accept_to_delivery_min'] = df['accept_to_delivery_min'].clip(upper=upper)
print('Max duration after clipping:', df['accept_to_delivery_min'].max())


Max duration after clipping: 233.0


In [18]:
# Cell 10a — Extract hour, day, month from accept_time
df['accept_hour'] = df['accept_time'].dt.hour # 0 to 23
df['accept_day'] = df['accept_time'].dt.day # 1 to 31
df['accept_month'] = df['accept_time'].dt.month # 1 to 12
print(df[['accept_time','accept_hour','accept_day','accept_month']].head(3))
# Cell 10b — Group hour into time-of-day buckets
def time_of_day(hour):
    if 0 <= hour < 6: return 'Night'
    elif 6 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else: return 'Night'
df['time_of_day'] = df['accept_hour'].apply(time_of_day)
print(df['time_of_day'].value_counts())

          accept_time  accept_hour  accept_day  accept_month
0 2023-06-04 11:05:00           11           4             6
1 2023-06-04 11:18:00           11           4             6
2 2023-06-03 10:13:00           10           3             6
time_of_day
Morning      708257
Afternoon    569363
Evening      197757
Night          8487
Name: count, dtype: int64


In [19]:
df['gps_drift'] = ((
(df['delivery_gps_lng'] - df['accept_gps_lng'])**2 +
(df['delivery_gps_lat'] - df['accept_gps_lat'])**2
) ** 0.5)
print(df['gps_drift'].describe().round(4))

count    1.483864e+06
mean     2.870000e-02
std      9.622000e-01
min      0.000000e+00
25%      1.020000e-02
50%      1.640000e-02
75%      2.530000e-02
max      1.256325e+02
Name: gps_drift, dtype: float64


In [20]:
df['delivery_speed_proxy'] = df['gps_drift'] / (df['accept_to_delivery_min'] + 1)
print('Speed proxy stats:')
print(df['delivery_speed_proxy'].describe().round(4))

Speed proxy stats:
count    1.483864e+06
mean     6.000000e-04
std      3.120000e-02
min      0.000000e+00
25%      1.000000e-04
50%      2.000000e-04
75%      4.000000e-04
max      3.139550e+01
Name: delivery_speed_proxy, dtype: float64


In [21]:
df['delivery_month'] = df['delivery_time'].dt.month
print(df['delivery_month'].value_counts().sort_index())

delivery_month
5      20913
6     183114
7     260279
8     316353
9     326823
10    376266
11       116
Name: count, dtype: int64


In [22]:
# Check unique city values (case-sensitive)
print(df['city'].unique())
print(df['city'].value_counts())


['Shanghai']
city
Shanghai    1483864
Name: count, dtype: int64


In [23]:
# Find all object (text) columns
text_cols = df.select_dtypes(include='object').columns
print('Text columns:', text_cols.tolist())
# ['city', 'accept_time', 'accept_gps_time', 'delivery_time', 'delivery_gps_time']

# Check each one
for col in text_cols:
    unique_vals = df[col].nunique()
    print(f'{col}: {unique_vals} unique values')

# city               → should be 1 (only "Hangzhou")
# accept_time        → will be many (timestamps)
# delivery_time      → will be many (timestamps)

Text columns: ['city', 'time_of_day']
city: 1 unique values
time_of_day: 4 unique values


In [24]:
print('Final dataset shape:', df.shape)
print()
print('Missing values left:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('New columns created:')
new_cols = ['gps_synced_accept', 'gps_synced_delivery',
'accept_to_delivery_min', 'accept_hour', 'accept_day',
'accept_month', 'time_of_day', 'gps_drift',
'delivery_speed_proxy', 'delivery_month']
print(new_cols)

Final dataset shape: (1483864, 27)

Missing values left:
Series([], dtype: int64)

New columns created:
['gps_synced_accept', 'gps_synced_delivery', 'accept_to_delivery_min', 'accept_hour', 'accept_day', 'accept_month', 'time_of_day', 'gps_drift', 'delivery_speed_proxy', 'delivery_month']


In [25]:
df.to_csv('delivery_sh_cleaned.csv', index=False)
print('File saved as: delivery_sh_cleaned.csv')
print(f'Total rows saved: {len(df):,}')

File saved as: delivery_hz_cleaned.csv
Total rows saved: 1,483,864
